# LlamaIndex + CoordiNode: PropertyGraphIndex

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/01_llama_index_property_graph.ipynb)

Demonstrates `CoordinodePropertyGraphStore` as a backend for LlamaIndex `PropertyGraphIndex`.

**What works right now:**
- `upsert_nodes` / `upsert_relations` — idempotent MERGE (safe to call multiple times)
- `get()` — look up nodes by ID or properties
- `get_triplets()` — all edges (wildcard) or filtered by relation type / entity name
- `get_rel_map()` — outgoing relations for a set of nodes (depth=1)
- `structured_query()` — arbitrary Cypher pass-through
- `delete()` — remove nodes by id or name
- `get_schema()` — live text schema of the graph

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). Installs pre-built wheel from PyPI (~30 sec).
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

pkgs = [
    "coordinode",
    "llama-index-graph-stores-coordinode",
    "llama-index-core",
    "nest_asyncio",
]
if IN_COLAB and not os.environ.get("COORDINODE_ADDR"):
    pkgs = ["coordinode-embedded"] + pkgs

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    check=True,
    timeout=120,
)

import nest_asyncio

nest_asyncio.apply()

print("SDK installed")


## Adapter for embedded mode

`LocalClient` (embedded engine) has the same `.cypher()` API as `CoordinodeClient`.
The `_EmbeddedAdapter` below adds the extra methods that the graph store expects
when it receives a pre-built `client=` object.

In [ ]:
class _EmbeddedAdapter:
    """Thin wrapper around LocalClient that adds CoordinodeClient-compatible methods."""

    def __init__(self, local_client):
        self._lc = local_client

    def cypher(self, query, params=None):
        return self._lc.cypher(query, params or {})

    def get_schema_text(self):
        lbls = self._lc.cypher("MATCH (n) UNWIND labels(n) AS lbl RETURN DISTINCT lbl ORDER BY lbl")
        rels = self._lc.cypher("MATCH ()-[r]->() RETURN DISTINCT type(r) AS t ORDER BY t")
        lines = ["Node labels:"]
        for r in lbls:
            lines.append(f"  - {r['lbl']}")
        lines.append("\nEdge types:")
        for r in rels:
            lines.append(f"  - {r['t']}")
        return "\n".join(lines)

    # Vector search not available in embedded mode — requires running CoordiNode server.

    def close(self):
        self._lc.close()


## Connect to CoordiNode

In [ ]:
import os, tempfile

# Persistent embedded DB path. Colab has /content which persists across cell
# reruns within a runtime session; locally fall back to the OS temp dir
# (portable across Linux/macOS/Windows). Override via COORDINODE_EMBEDDED_PATH.
COORDINODE_EMBEDDED_PATH = os.environ.get(
    "COORDINODE_EMBEDDED_PATH",
    "/content/coordinode-demo.db"
    if os.path.isdir("/content")
    else os.path.join(tempfile.gettempdir(), "coordinode-demo.db"),
)

if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    _cc = CoordinodeClient(COORDINODE_ADDR)
    if not _cc.health():
        _cc.close()
        raise RuntimeError(f"Health check failed for {COORDINODE_ADDR}")
    print(f"Connected to {COORDINODE_ADDR}")
    client = _cc
else:
    # No explicit server — use the embedded in-process engine backed by a file
    # so the graph persists across cell reruns and between sibling demo
    # notebooks within the same runtime.
    try:
        from coordinode_embedded import LocalClient
    except ImportError as exc:
        raise RuntimeError(
            "coordinode-embedded is not installed. "
            "In Colab, rerun the install cell above. "
            "Locally, install from source: "
            "pip install 'git+https://github.com/structured-world/coordinode-python.git#subdirectory=coordinode-embedded' "
            "(requires Rust toolchain, ~5 min build). "
            "Alternatively start a CoordiNode server and set COORDINODE_ADDR."
        ) from exc

    _lc = LocalClient(COORDINODE_EMBEDDED_PATH)
    # Wrap in _EmbeddedAdapter so CoordinodeGraph/PropertyGraphStore can call
    # get_schema_text() — LocalClient has .cypher() only.
    client = _EmbeddedAdapter(_lc)
    print(f"Using embedded LocalClient at {COORDINODE_EMBEDDED_PATH}")


## Create the property graph store

Pass the pre-built `client=` object so the store doesn't open a second connection.

In [ ]:
from llama_index.graph_stores.coordinode import CoordinodePropertyGraphStore
from llama_index.core.graph_stores.types import EntityNode, Relation

store = CoordinodePropertyGraphStore(client=client)
print("Connected. Schema:")
print(store.get_schema()[:300])

## 1. Upsert nodes and relations

Each node gets a unique tag so this notebook can run multiple times without conflicts.

In [ ]:
import uuid

tag = uuid.uuid4().hex

nodes = [
    EntityNode(label="Person", name=f"Alice-{tag}", properties={"role": "researcher", "field": "AI"}),
    EntityNode(label="Person", name=f"Bob-{tag}", properties={"role": "engineer", "field": "ML"}),
    EntityNode(label="Topic", name=f"GraphRAG-{tag}", properties={"domain": "knowledge graphs"}),
]
store.upsert_nodes(nodes)
print("Upserted nodes:", [n.name for n in nodes])

alice, bob, graphrag = nodes
relations = [
    Relation(label="RESEARCHES", source_id=alice.id, target_id=graphrag.id, properties={"since": 2023}),
    Relation(label="COLLABORATES", source_id=alice.id, target_id=bob.id),
    Relation(label="IMPLEMENTS", source_id=bob.id, target_id=graphrag.id),
]
store.upsert_relations(relations)
print("Upserted relations:", [r.label for r in relations])

## 2. get_triplets — all edges from a node (wildcard)

In [ ]:
triplets = store.get_triplets(entity_names=[f"Alice-{tag}"])
print(f"Triplets for Alice-{tag}:")
for src, rel, dst in triplets:
    print(f"  {src.name}  --[{rel.label}]-->  {dst.name}")

## 3. get_rel_map — relations for a set of nodes

In [ ]:
found_alice = store.get(properties={"name": f"Alice-{tag}"})
rel_map = store.get_rel_map(found_alice, depth=1, limit=20)
print(f"Rel map for Alice-{tag} ({len(rel_map)} rows):")
for src, rel, dst in rel_map:
    print(f"  {src.name}  --[{rel.label}]-->  {dst.name}")

## 4. structured_query — arbitrary Cypher

In [ ]:
rows = store.structured_query(
    "MATCH (p:Person)-[r:RESEARCHES]->(t:Topic)"
    " WHERE p.name STARTS WITH $prefix"
    " RETURN p.name AS person, t.name AS topic, r.since AS since",
    param_map={"prefix": f"Alice-{tag}"},
)
print("Query result:", rows)

## 5. get_schema

In [ ]:
schema = store.get_schema()
print(schema)

## 6. Idempotency — double upsert must not duplicate edges

In [ ]:
store.upsert_relations(relations)  # second call — should still be exactly 1 edge
rows = store.structured_query(
    "MATCH (a {name: $src})-[r:RESEARCHES]->(b {name: $dst}) RETURN count(r) AS cnt",
    param_map={"src": f"Alice-{tag}", "dst": f"GraphRAG-{tag}"},
)
print("Edge count after double upsert (expect 1):", rows[0]["cnt"])

## Cleanup

In [ ]:
store.delete(entity_names=[f"Alice-{tag}", f"Bob-{tag}", f"GraphRAG-{tag}"])
print("Cleaned up")
store.close()
client.close()  # injected client — owned by caller